# 3.4 Bayes' Theorem

**Goal:** update uncertainty after evidence, distinguish a likelihood from a posterior, and build a small probabilistic classifier.

**Prerequisites:** conditional probability, the chain rule, total probability, and logarithms. **Runtime:** Python 3.10+, standard library only. All datasets are synthetic. Saved outputs are included; run cells in order.

## Contents
1. Bayes' theorem and derivation
2. Prior probability
3. Likelihood
4. Evidence
5. Posterior probability
6. Bayesian updating
7. Uncertainty and posterior prediction
8. Worked Naive Bayes classifier
9. Practice and solutions

## 1. Bayes' Theorem

The multiplication rule expresses the same joint probability in two ways:
$$P(H\cap E)=P(E\mid H)P(H)=P(H\mid E)P(E).$$
If $P(E)>0$, rearranging gives
$$P(H\mid E)=\frac{P(E\mid H)P(H)}{P(E)}.$$
For mutually exclusive, exhaustive hypotheses $H_1,\ldots,H_k$ with positive prior mass,
$$P(H_i\mid E)=\frac{P(E\mid H_i)P(H_i)}{\sum_jP(E\mid H_j)P(H_j)}.$$
The denominator comes from the law of total probability. Bayes' theorem does not create missing evidence or validate assumptions; it specifies how coherent probabilities relate.

## 2. Prior Probability

The prior $P(H)$ expresses uncertainty **before the current evidence**. It can represent a population base rate, earlier data, substantive information, or a deliberately chosen model assumption. “Before” is relative to the update: yesterday's posterior can be today's prior for genuinely new evidence.

A prior is not necessarily arbitrary, equal across hypotheses, or permanent. State its source and check sensitivity where decisions depend strongly on it. A hypothesis with prior exactly zero cannot gain posterior mass from an ordinary finite-likelihood update, so zero is a strong modeling commitment.

## 3. Likelihood

The likelihood evaluates the probability of the observed evidence under each proposed hypothesis: $P(E\mid H)$. For fixed evidence it is a function of H (or a parameter), not a probability distribution over hypotheses. Its values need not sum to 1 across hypotheses.

For continuous observations, likelihoods are often density values, which can exceed 1; the same Bayes normalization idea applies with the appropriate densities. Here examples use finite events.

## 4. Evidence

The evidence or marginal probability is
$$P(E)=\sum_jP(E\mid H_j)P(H_j).$$
It normalizes the unnormalized scores into posterior probabilities. It must account for all hypotheses in the modeled partition. If every score is zero, the observation is impossible under that model; division by zero is a signal to inspect the model or data, not choose a posterior by convenience.

## 5. Posterior Probability

The posterior $P(H\mid E)$ is the updated uncertainty **after** observing E. It combines likelihood and prior, normalized by evidence. High $P(E\mid H)$ alone does not establish high $P(H\mid E)$: alternative explanations and base rates matter.

### Worked example: an alert and a rare defect

Suppose 1% of manufactured items are defective. A screening system flags 90% of defective items and 5% of nondefective items. Then
$$P(\text{flag})=0.90(0.01)+0.05(0.99)=0.0585,$$
$$P(\text{defect}\mid\text{flag})=0.009/0.0585=2/13\approx0.15385.$$

Among a hypothetical 10000 items with those proportions, 90 defective and 495 nondefective items would be flagged: 90/585 are actually defective. A 90% detection probability is not a 90% probability of defect given a flag. These counts illustrate the specified model, not observed factory performance.

In [1]:
from fractions import Fraction
import math
prior_defect = Fraction(1, 100)
flag_if_defect = Fraction(9, 10)
flag_if_good = Fraction(1, 20)
evidence = flag_if_defect*prior_defect + flag_if_good*(1-prior_defect)
posterior_defect = flag_if_defect*prior_defect/evidence
print("P(flag):", evidence, float(evidence))
print("P(defect | flag):", posterior_defect, float(posterior_defect))
print("Model counts among 10000: flagged defective=90; flagged nondefective=495")
assert posterior_defect == Fraction(2, 13)

P(flag): 117/2000 0.0585
P(defect | flag): 2/13 0.15384615384615385
Model counts among 10000: flagged defective=90; flagged nondefective=495


### Odds form and base-rate sensitivity

For two hypotheses H and $H^c$ with nonzero denominators,
$$\frac{P(H\mid E)}{P(H^c\mid E)}=\frac{P(H)}{P(H^c)}\times\frac{P(E\mid H)}{P(E\mid H^c)}.$$
The last factor is a likelihood ratio (also a Bayes factor for these specified hypotheses). Evidence multiplies prior odds. The same screening characteristics can produce different posterior probabilities in populations with different defect prevalence.

A low posterior after a flag need not mean the screen is useless. Decisions also depend on inspection cost, missed-defect harm, and capacity. A probability and a decision threshold are different objects.

In [2]:
for prevalence in [0.001, 0.01, 0.10]:
    p_flag = 0.9*prevalence + 0.05*(1-prevalence)
    p_defect_given_flag = 0.9*prevalence/p_flag
    print(f"Prior prevalence={prevalence:.3f}, posterior after flag={p_defect_given_flag:.4f}")
prior_odds = prior_defect/(1-prior_defect)
posterior_odds = prior_odds*(flag_if_defect/flag_if_good)
assert posterior_odds/(1+posterior_odds) == posterior_defect

Prior prevalence=0.001, posterior after flag=0.0177
Prior prevalence=0.010, posterior after flag=0.1538
Prior prevalence=0.100, posterior after flag=0.6667


## 6. Bayesian Updating

Sequential updating uses the previous posterior as the next prior. For observations $E_1,E_2$,
$$P(H\mid E_1,E_2)\propto P(E_2\mid H,E_1)P(H\mid E_1).$$
Only if the observations are conditionally independent given H can the likelihood reduce to $P(E_2\mid H)$. Repeated alerts from the same sensor or duplicated features can be dependent. Multiplying the same evidence twice as though it were new can create false confidence.

**Worked example:** a coin has one of two possible head probabilities, 0.5 or 0.8, with equal prior probabilities. Given the coin's parameter, tosses are assumed independent. After one head, the posterior probability of the 0.8 coin is $0.8/(0.5+0.8)=8/13$. A second head favors it further; a tail shifts belief back toward the 0.5 coin.

The code uses exact fractions to show that sequential updates match a one-shot update using the full likelihood product. The model deliberately excludes every other possible coin bias: its posterior only compares the specified hypotheses.

In [3]:
def update_discrete(prior, likelihood):
    if not prior or set(prior) != set(likelihood):
        raise ValueError("Prior and likelihood must cover the same hypotheses")
    if any(p < 0 for p in prior.values()) or sum(prior.values()) != 1:
        raise ValueError("Prior must have nonnegative probabilities summing exactly to one")
    if any(p < 0 or p > 1 for p in likelihood.values()):
        raise ValueError("This finite-event example requires likelihood probabilities in [0, 1]")
    scores = {h: prior[h]*likelihood[h] for h in prior}
    evidence = sum(scores.values())
    if evidence == 0:
        raise ValueError("Evidence has probability zero under the model")
    return {h: score/evidence for h, score in scores.items()}

head_p = {"fair": Fraction(1, 2), "head_biased": Fraction(4, 5)}
initial = {h: Fraction(1, 2) for h in head_p}
posterior = initial.copy()
observations = ["H", "H", "T"]
for observation in observations:
    likelihood = {h: p if observation == "H" else 1-p for h, p in head_p.items()}
    posterior = update_discrete(posterior, likelihood)
    print("After", observation, ":", {h: round(float(p), 6) for h, p in posterior.items()})
full_likelihood = {h: p*p*(1-p) for h, p in head_p.items()}
batch = update_discrete(initial, full_likelihood)
assert posterior == batch
assert sum(posterior.values()) == 1

After H : {'fair': 0.384615, 'head_biased': 0.615385}
After H : {'fair': 0.280899, 'head_biased': 0.719101}
After T : {'fair': 0.494071, 'head_biased': 0.505929}


## 7. Uncertainty Estimation and Posterior Prediction

A posterior can spread probability across multiple hypotheses instead of selecting only the most likely one. For the next toss, the posterior predictive probability averages over this remaining uncertainty:
$$P(\text{next H}\mid D)=\sum_hP(\text{next H}\mid h,D)P(h\mid D).$$
Under the conditional-independent toss model, the first term is the coin's head probability $p_h$. This distinguishes **uncertainty about the parameter** from the remaining randomness of a future toss, even if the parameter were known.

For this discrete model, parameter variance is $\sum_hw_h(p_h-\bar p)^2$. Future Bernoulli variance has the decomposition
$$\bar p(1-\bar p)=\sum_hw_hp_h(1-p_h)+\sum_hw_h(p_h-\bar p)^2.$$
The first term averages outcome variation at a known parameter; the second reflects posterior parameter uncertainty. This elementary illustration is not a complete account of uncertainty in real ML systems, which may have misspecified models and shifted data.

Posterior confidence is conditional on the hypothesis set and likelihood assumptions. A sharp posterior over two wrong hypotheses is not assurance that either model is correct.

In [4]:
predictive = sum(posterior[h]*head_p[h] for h in head_p)
parameter_variance = sum(posterior[h]*(head_p[h]-predictive)**2 for h in head_p)
conditional_outcome_variance = sum(posterior[h]*head_p[h]*(1-head_p[h]) for h in head_p)
print("Posterior predictive P(next H):", float(predictive))
print("Posterior variance of head-probability parameter:", float(parameter_variance))
print("Conditional outcome-variance component:", float(conditional_outcome_variance))
print("Total next-toss variance:", float(predictive*(1-predictive)))
assert predictive*(1-predictive) == conditional_outcome_variance+parameter_variance

Posterior predictive P(next H): 0.6517786561264822
Posterior variance of head-probability parameter: 0.022496836382383728
Conditional outcome-variance component: 0.20446640316205533
Total next-toss variance: 0.22696323954443906


## 8. AI/ML application: a Bernoulli Naive Bayes classifier

For binary features $x_j\in\{0,1\}$, the model is
$$P(C=c\mid x)\propto P(C=c)\prod_j\theta_{cj}^{x_j}(1-\theta_{cj})^{1-x_j},$$
where $\theta_{cj}=P(X_j=1\mid C=c)$. Both present and absent features contribute. The “naive” assumption is conditional independence of features given the class, not marginal independence.

The toy training rows below record whether a message contains “offer” and “meeting.” Class priors are training frequencies. Add-one smoothing gives $\theta_{cj}=(\text{positive count}+1)/(\text{class size}+2)$, preventing zero probabilities for unseen binary outcomes. These smoothed point estimates define this fitted model; it is not a full posterior over all parameters.

The implementation sums log probabilities and subtracts the largest log score before exponentiating, avoiding direct products of tiny numbers. Normalizing the scores produces probabilities under this model. They are not automatically calibrated real-world probabilities, especially with correlated features or shifted class prevalences. A genuine classifier needs separate evaluation data; this example demonstrates the calculation.

In [5]:
training = [
    ((1, 0), "spam"), ((1, 0), "spam"), ((1, 1), "spam"), ((0, 0), "spam"),
    ((0, 1), "ham"), ((0, 1), "ham"), ((1, 1), "ham"), ((0, 0), "ham"),
]
classes = sorted({label for _, label in training})
class_rows = {c: [x for x, label in training if label == c] for c in classes}
priors = {c: len(class_rows[c])/len(training) for c in classes}
theta = {c: [(sum(x[j] for x in class_rows[c])+1)/(len(class_rows[c])+2)
             for j in range(2)] for c in classes}

def predict_proba(features):
    if len(features) != 2 or any(value not in (0, 1) for value in features):
        raise ValueError("Expected two binary word-presence features")
    log_scores = {}
    for c in classes:
        log_scores[c] = math.log(priors[c])+sum(
            math.log(p) if value else math.log1p(-p)
            for value, p in zip(features, theta[c]))
    offset = max(log_scores.values())
    scores = {c: math.exp(score-offset) for c, score in log_scores.items()}
    total = sum(scores.values())
    return {c: score/total for c, score in scores.items()}

print("Class priors:", priors)
print("Smoothed feature probabilities [offer, meeting]:", theta)
for features in [(1, 0), (0, 1), (1, 1), (0, 0)]:
    probabilities = predict_proba(features)
    print(features, {c: round(p, 6) for c, p in probabilities.items()})
    assert math.isclose(sum(probabilities.values()), 1)
    assert all(0 <= p <= 1 for p in probabilities.values())
assert math.isclose(predict_proba((1, 0))["spam"], 0.8)

Class priors: {'ham': 0.5, 'spam': 0.5}
Smoothed feature probabilities [offer, meeting]: {'ham': [0.3333333333333333, 0.6666666666666666], 'spam': [0.6666666666666666, 0.3333333333333333]}
(1, 0) {'ham': 0.2, 'spam': 0.8}
(0, 1) {'ham': 0.8, 'spam': 0.2}
(1, 1) {'ham': 0.5, 'spam': 0.5}
(0, 0) {'ham': 0.5, 'spam': 0.5}


## Practice

1. A defect prevalence is 2%, detection probability 80%, and false-flag probability 10%. Find the probability of defect given a flag.
2. Identify the prior, likelihood, evidence, and posterior in question 1.
3. With equal priors for a fair coin and an always-heads coin, what is the posterior probability of the always-heads coin after one head? After observing a tail next?
4. Why can a prior probability exactly zero be problematic?
5. When is it legitimate to multiply two observation likelihoods in sequential updating?
6. For coin hypotheses $p=0.2$ and $p=0.8$ with posterior weights 0.25 and 0.75, find the next-head predictive probability.
7. Does a Naive Bayes likelihood use only present features in the Bernoulli model above?
8. Does a normalized posterior prove that the probabilities are well calibrated?

## Solutions

1. Evidence $0.8(0.02)+0.1(0.98)=0.114$. Posterior $0.016/0.114=8/57\approx0.14035$.
2. Prior $P(D)=0.02$; likelihoods $P(F\mid D)=0.8$ and $P(F\mid D^c)=0.1$; evidence $P(F)=0.114$; posterior $P(D\mid F)=8/57$.
3. After H, posterior $1/(1+0.5)=2/3$. After a subsequent T, the always-heads hypothesis has zero likelihood, so its posterior becomes 0 under this idealized model.
4. Ordinary updating cannot restore posterior mass to a zero-prior hypothesis even when new data favor it relative to alternatives. Use zero only for a deliberate impossibility assumption.
5. When they are conditionally independent given the hypothesis; otherwise use $P(E_2\mid H,E_1)$ rather than $P(E_2\mid H)$.
6. $0.25(0.2)+0.75(0.8)=0.65$, averaging rather than choosing the most probable parameter alone.
7. No. Both presence and absence contribute factors; ignoring absences would change the model.
8. No. Normalization only ensures the model scores sum to 1. Calibration requires appropriate empirical evaluation and depends on model/data validity.

**Readiness check:** separate $P(E\mid H)$ from $P(H\mid E)$, include base rates, avoid double-counting evidence, and distinguish posterior prediction from selecting one hypothesis.

**Next:** [3.5 Counting Techniques](5_Counting-Techniques.ipynb).